# Implementation plan — summarization rerun

## Summary
Create this operational notebook to rerun summarization recursively over
`PROJ_ROOT/assets/md.json.semantics.summary` using the updated `DOMClass`.

## Configuration and processing
- Configure `PROJ_ROOT/.env` with `OLLAMA_HOST=http://172.27.74.16:11434` and `OLLAMA_SUMMARY_MODEL=qwen3.8:27b`, preserving unrelated settings.
- Resolve the project root from the environment or nearest parent containing `pyproject.toml`; load its `.env` before importing processing modules.
- Share one Ollama async client and pass its configured model explicitly to every DOMClass for both text and images; use ephemeral Chroma storage.
- Discover sorted Markdown files recursively and check the remote server lists the configured model before processing.
- Process with bounded asyncio document workers and fresh DOMClass instances, calling `setup()` then `await textualize()` even when semantics already exist.
- Replace `_semantics.json` through textualize; retain cached ASTs and generate missing AST artifacts through setup.
- Report per-file status, elapsed time, output path, and errors; continue after failures, propagate cancellation, and display final results.
- Keep saved outputs cleared; do not run the corpus during notebook creation.

## Validation
Validate notebook structure and code compilation. Use temporary fixtures and mocked
providers to verify text/image configuration forwarding, summary replacement, AST
preservation, failure isolation, and missing/empty inputs and provider failures.

## Assumptions
Both text and images use the configured Ollama model. Existing summaries are replaced.
Embedding and ingestion are excluded; existing model and embedding notebook edits remain untouched.


## Configuration
Run cells in order. The processing cell overwrites summary artifacts beside each Markdown
file. Cached ASTs are reused. The notebook uses the host and model from the project's `.env`.

Set limits in the environment or `.env` before running this cell. Defaults are **4 document
workers, 8 shared model requests, and 32 traversal tasks per document**. If DOM modules have
already been imported with different limits, restart the kernel rather than reloading them.


In [ ]:
import asyncio
import json
import os
import sys
from pathlib import Path
from time import perf_counter

from dotenv import dotenv_values, load_dotenv


def resolve_project_root():
    configured = os.getenv("PROJ_ROOT")
    if configured:
        root = Path(configured).expanduser().resolve()
        if not (root / "pyproject.toml").is_file():
            raise FileNotFoundError(f"Invalid PROJ_ROOT (missing pyproject.toml): {root}")
        return root
    cwd = Path.cwd().resolve()
    for root in (cwd, *cwd.parents):
        if (root / "pyproject.toml").is_file():
            return root
    raise FileNotFoundError("Cannot locate PROJ_ROOT containing pyproject.toml")


PROJ_ROOT = resolve_project_root()
ENV_FILE = PROJ_ROOT / ".env"
if not ENV_FILE.is_file():
    raise FileNotFoundError(f"Missing configuration: {ENV_FILE}")
load_dotenv(ENV_FILE, override=False)
env_config = dotenv_values(ENV_FILE)
OLLAMA_HOST = env_config.get("OLLAMA_HOST")
SUMMARY_MODEL = env_config.get("OLLAMA_SUMMARY_MODEL")
if not OLLAMA_HOST or not SUMMARY_MODEL:
    raise ValueError("Set OLLAMA_HOST and OLLAMA_SUMMARY_MODEL in PROJ_ROOT/.env")
# Keep module-level clients consistent with this notebook's explicit configuration.
os.environ["OLLAMA_HOST"] = OLLAMA_HOST
if str(PROJ_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJ_ROOT))

# Limits must be applied before DOM modules are imported. Environment/.env overrides win.
from utils.summary_asyncio import configure_summary_limits, SummaryAsyncClient
SUMMARY_LIMITS = configure_summary_limits()
DOCUMENT_CONCURRENCY = SUMMARY_LIMITS["RIBOSOME_DOCUMENT_CONCURRENCY"]

import chromadb
import pandas as pd
from IPython.display import display
from ribosome.core.dom.model import DOMClass

INPUT_DIR = PROJ_ROOT / "assets" / "md.json.semantics.summary"
CHECKPOINT_PATH = INPUT_DIR / ".summary-checkpoint.json"
RESUME = True  # Set False to regenerate every document and replace the checkpoint.
ollama_client = SummaryAsyncClient(host=OLLAMA_HOST)
db_client = chromadb.EphemeralClient()
print(f"Input: {INPUT_DIR}\nOllama: {OLLAMA_HOST}\nText and image model: {SUMMARY_MODEL}")

print(f"Effective concurrency: {SUMMARY_LIMITS}")


## Discovery and provider preflight
No summaries are generated in this step. The configured model must already be installed on the server.


In [ ]:
def discover_markdown(input_dir):
    input_dir = Path(input_dir)
    if not input_dir.is_dir():
        raise FileNotFoundError(f"Markdown input directory does not exist: {input_dir}")
    files = sorted(path for path in input_dir.rglob("*")
                   if path.is_file() and path.suffix.lower() == ".md")
    if not files:
        raise ValueError(f"No Markdown files found under: {input_dir}")
    return files


async def check_model(client, model, host):
    try:
        response = await client.list()
    except Exception as exc:
        raise RuntimeError(f"Cannot list Ollama models at {host}: {exc}") from exc
    available = {item.model for item in response.models}
    if model not in available:
        raise RuntimeError(f"Model {model!r} is unavailable at {host}; available: {sorted(available)}")
    print(f"Provider ready: {model}")


md_files = discover_markdown(INPUT_DIR)
print(f"Markdown files: {len(md_files)}")
await check_model(ollama_client, SUMMARY_MODEL, OLLAMA_HOST)


## Rerun summarization
This cell replaces semantics artifacts for documents still needing processing.
Progress is atomically checkpointed after each document and on cancellation in
`INPUT_DIR/.summary-checkpoint.json`. With `RESUME = True`, rerunning cells after a
kernel restart skips successful documents whose Markdown, cached AST, summary output,
model, and host match the checkpoint. Failed or interrupted documents restart from
the beginning; progress within an individual document is not saved. Set `RESUME = False`
to regenerate all documents. Keep the checkpoint alongside the corpus. Each document starts from its cached
AST, not its old semantics. Failures are recorded and later documents continue.

Setup runs in bounded background threads while Ollama requests run on the event loop.
Interrupting cancels model work and waits for already-started setup threads to finish;
a Pandoc conversion in a thread cannot be stopped immediately.

Two progress bars show completed documents and typed AST nodes. Counters update on every
node; the display refreshes every **250ms**, with a final flush. Text and image requests
share the same semaphore. Request metrics distinguish client latency from Ollama's returned
load, prompt evaluation, and generation timings; peak requests measures client overlap,
not the server's inference parallelism.


In [ ]:
from utils.summary_asyncio import summarize_documents

# Shared client, bounded workers, threaded setup, and periodically rendered progress.
ollama_client.reset_metrics()
batch_start = perf_counter()
results = await summarize_documents(
    md_files, ollama_client, SUMMARY_MODEL, db_client,
    concurrency=DOCUMENT_CONCURRENCY, progress_interval=0.25,
    checkpoint_path=CHECKPOINT_PATH, resume=RESUME, checkpoint_host=OLLAMA_HOST,
)
batch_elapsed_seconds = perf_counter() - batch_start
batch_metrics = ollama_client.metrics.snapshot(wall_seconds=batch_elapsed_seconds)


## Results
Review failed documents before rerunning the processing cell. Unchanged successful documents are skipped on resume and marked `resumed=True`.
Their elapsed times and request metrics come from the saved run; batch metrics cover only
the current run. Changes to images or summarizer code require `RESUME = False`.


In [ ]:
results_df = pd.DataFrame(results)
display(results_df)
print(results_df["status"].value_counts().to_string())
print(f"Summed document processing time: {results_df['elapsed_seconds'].sum():.3f}s")
failures = results_df.loc[results_df["status"] == "failed"]
if not failures.empty:
    print("Failed documents:")
    display(failures[["file", "error"]])

print(f"Batch wall time: {batch_elapsed_seconds:.3f}s")

display(pd.DataFrame([batch_metrics]))


## Benchmark without replacing corpus outputs
Run the following command from the project root in a terminal. It reads fixed bounded
samples from cached ASTs (text, images, and deep nesting), warms them once, compares the
original scheduling behavior with the inline-singleton scheduler, and reports medians of
three measured runs. Summary JSON and reports go into a temporary directory by default.
The benchmark never calls `setup()` or changes corpus artifacts. Each pass has a
300-second time limit; use `--timeout-seconds` to adjust it. A failed warmup stops
the benchmark and saves diagnostics instead of reporting a misleading speedup.

```sh
.venv/bin/python utils/benchmark_summary_asyncio.py --live
```

For an offline scheduling demonstration with simulated request latency:

```sh
.venv/bin/python utils/benchmark_summary_asyncio.py
```

Server tuning must be done on the machine running Ollama, not in this notebook's `.env`.
Inspect `ollama --version`, `ollama ps`, GPU memory, and the service's environment first.
For a systemd service, add `Environment="OLLAMA_NUM_PARALLEL=2"` under `[Service]` in
`systemctl edit ollama.service`, then reload systemd and restart Ollama. Use the equivalent
server environment setting for Docker or Windows. Test 1, 2, then 4 if memory permits.
After each restart run the benchmark with request slots matching server parallelism and
then twice that value (capped at 8), e.g. `--request-limit 2` and `--request-limit 4`.
Keep the model, generation settings, and AST samples unchanged. Choose the best median
throughput, preferring lower concurrency when results are within 5%. Record the server
parallelism with `--server-parallel 2`; this records the value, it does not configure it.
See [the implementation plan](../docs/summary-asyncio-acceleration-plan.md) and
[Ollama's concurrency documentation](https://docs.ollama.com/faq#how-does-ollama-handle-concurrent-requests).
